In [0]:
# Read secret from Azure Key Vault through Databricks Secret Scope

dbutils.secrets.listScopes()
storage_key = dbutils.secrets.get(
    scope="kv_scope",
    key="storage-key"
)

# Configure Spark
spark.conf.set(
    "fs.azure.account.key.ecomdatalake345.dfs.core.windows.net",
    storage_key
)

# List files
display(
    dbutils.fs.ls(
        "abfss://bronze@ecomdatalake345.dfs.core.windows.net/"
    )
)


In [0]:
# -----------------------------------------
# Bronze Layer - Customer Data Ingestion
# -----------------------------------------

from pyspark.sql import functions as F

# Source and Target Paths
source_path = "abfss://sales-view-devst@ecomdatalake345.dfs.core.windows.net/raw_customer"
bronze_path = "abfss://bronze@ecomdatalake345.dfs.core.windows.net/bronze_customer"

# Read Raw Customer Table
df = spark.read.format("csv").option("header" , "true").option("inferSchema", "true").load(source_path)

# Add Bronze Metadata Columns new change
bronze_df = (
    df.withColumn("bronze_ingestion_ts", F.current_timestamp())
      .withColumn("bronze_source_file", F.col("_metadata.file_path"))
)

# Write to Bronze Delta Layer
(
    bronze_df.write
             .format("delta")
             .mode("overwrite")
             .option("overwriteSchema","true")
             .save(bronze_path)
)


# Validation
record_count = spark.read.format('delta').load(bronze_path).count()

print(f"Bronze Load Completed Successfully")
print(f"Records Loaded: {record_count}")

display(bronze_df)